# Demo for loading and using pretrained models

In [ ]:
import sys, os 
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..') ## set working dir to root
import numpy as np 
import h5py
import pickle
import matplotlib.pyplot as plt
import tensorflow as tf
print('TF v: %s' % tf.__version__)
tf.config.set_visible_devices([], 'GPU') # do not use any GPU

if 'src' not in sys.path:
    sys.path.append('src')

import utils

In [ ]:
def expand_dims_maybe_(x):
    if np.ndim(x) == 2:
        x = np.expand_dims(np.expand_dims(x, axis=0), axis=-1)
    elif np.ndim(x) == 3:
        x = np.expand_dims(x, axis=0) if np.shape(x)[-1] == 1 else np.expand_dims(x, axis=-1)
    return x

@tf.function
def scaleclip_batch_tf(x):
    x = x/tf.reduce_max(x, axis=(1,2), keepdims=True)
    x = tf.clip_by_value(x, clip_value_min=-0.2, clip_value_max=1.)
    return x

@tf.function
def predict_seg_tf(model, x_t, ytrue_t):
    x_t = scaleclip_batch_tf(x_t)
    ypred_t = model(x_t, training=False)
    return x_t, ytrue_t, ypred_t

def predict_seg(model, x, ytrue):
    x = expand_dims_maybe_(x)
    x_t = tf.convert_to_tensor(x)
    ytrue_t = tf.convert_to_tensor(ytrue)
    x_t, ytrue_t, ypred_t = predict_seg_tf(model, x_t, ytrue_t)
    x = np.squeeze(x_t.numpy())
    ytrue = np.squeeze(ytrue_t.numpy())    
    ypred = np.squeeze(ypred_t.numpy())
    ypred = np.argmax(ypred, axis=-1)
    return x, ytrue, ypred


@tf.function
def scaleclip_batch_tf(x):
    x = x/tf.reduce_max(x, axis=(1,2), keepdims=True)
    x = tf.clip_by_value(x, clip_value_min=-0.2, clip_value_max=1.)
    return x

@tf.function
def predict_tf(model, x_t, ytrue_t):
    x_t = scaleclip_batch_tf(x_t)
    ytrue_t = scaleclip_batch_tf(ytrue_t)
    ypred_t = model(x_t, training=False)
    ypred_t = scaleclip_batch_tf(ypred_t)
    return x_t, ytrue_t, ypred_t

def predict(model, x, ytrue):
    x = expand_dims_maybe_(x)
    ytrue = expand_dims_maybe_(ytrue)
    x_t = tf.convert_to_tensor(x)
    ytrue_t = tf.convert_to_tensor(ytrue)
    x_t, ytrue_t, ypred_t = predict_tf(model, x_t, ytrue_t)
    x = np.squeeze(x_t.numpy())
    ytrue = np.squeeze(ytrue_t.numpy())    
    ypred = np.squeeze(ypred_t.numpy())
    return x, ytrue, ypred

In [ ]:
mpm_obj = utils.Manage_Pretrained_Models()
list_tasks = mpm_obj.list_task_str
print(f'Task strings: {list_tasks}')

## Segmentation task example

1. On a sample from test set

In [ ]:
## Example with seg_ss32,vc
task_str = 'seg_ss32,vc'
model = mpm_obj.load_model(task_str=task_str)

## load task and dataset relevant hyperparameters
datasets_parent_dir = '/home/firat/docs/dlbirhoui/parsed_data2'
hpobj = utils.load_model_hyperparameters(task_str=task_str, datasets_parent_dir=datasets_parent_dir)

fname = hpobj.fname_h5
in_key = hpobj.in_key
out_key = hpobj.out_key

## predict and evaluate a sample 
metrics_computer = utils.Metrics_Segmentation(num_classes=3)
fname_sample_data = os.path.join('data', 'sample_data', os.path.basename(fname).replace('h5', 'p'))
!git lfs pull -I {fname_sample_data}
with open(fname_sample_data, 'rb') as fh:
    d = pickle.load(fh)
im = d[in_key]
gt = d[out_key]
x, ytrue, ypred = predict_seg(model, x=im, ytrue=gt)
d_res = metrics_computer.compute(ypred=ypred, ytrue=ytrue)
print(f"IoU scores for BG: {d_res['iou'][0]:.2f}, vessel: {d_res['iou'][1]:.2f}, skincurve: {d_res['iou'][2]:.2f}")

ims = (x, ypred, ytrue)
titles = ('input', 'prediction', 'target')
fig, axs = plt.subplots(nrows=1,ncols=3, figsize=(8*3,8*1), dpi=200)
for i in range(3):
    axs[i].imshow(ims[i], cmap='gray')
    axs[i].set_title(titles[i])
    axs[i].set_axis_off()
plt.show()

2. On a random test set sample (requires public dataset to be downloaded)

In [ ]:
## Example with seg_ss32,vc
task_str = 'seg_ss32,vc'
model = mpm_obj.load_model(task_str=task_str)

## load task and dataset relevant hyperparameters
datasets_parent_dir = '/home/firat/docs/dlbirhoui/parsed_data2'
hpobj = utils.load_model_hyperparameters(task_str=task_str, datasets_parent_dir=datasets_parent_dir)

fname = hpobj.fname_h5
inds_test = hpobj.inds_test
inds_train = hpobj.inds_train
inds_val = hpobj.inds_val
in_key = hpobj.in_key
out_key = hpobj.out_key

metrics_computer = utils.Metrics_Segmentation(num_classes=3)

## predict and evaluate a random sample
i = np.random.choice(inds_test)
with h5py.File(fname, 'r') as fh:
    im = fh[in_key][i,...]
    gt = fh[out_key][i,...]
x, ytrue, ypred = predict_seg(model, x=im, ytrue=gt)
d_res = metrics_computer.compute(ypred=ypred, ytrue=ytrue)
print(f"IoU scores for BG: {d_res['iou'][0]:.2f}, vessel: {d_res['iou'][1]:.2f}, skincurve: {d_res['iou'][2]:.2f}")

ims = (x, ypred, ytrue)
titles = ('input', 'prediction', 'target')
fig, axs = plt.subplots(nrows=1,ncols=3, figsize=(8*3,8*1), dpi=200)
for i in range(3):
    axs[i].imshow(ims[i], cmap='gray')
    axs[i].set_title(titles[i])
    axs[i].set_axis_off()
plt.show()

# Image translation task example

1. On a sample from test set

In [ ]:
## Example with swfd_lv128,sc
task_str = 'swfd_lv128,sc'
model = mpm_obj.load_model(task_str=task_str)

## load task and dataset relevant hyperparameters
datasets_parent_dir = '.'
hpobj = utils.load_model_hyperparameters(task_str=task_str, datasets_parent_dir=datasets_parent_dir)

fname = hpobj.fname_h5
in_key = hpobj.in_key
out_key = hpobj.out_key

## predict and evaluate a sample 
metrics_computer = utils.Metrics_Translation()
fname_sample_data = os.path.join('data', 'sample_data', os.path.basename(fname).replace('h5', 'p'))
!git lfs pull -I {fname_sample_data}
with open(fname_sample_data, 'rb') as fh:
    d = pickle.load(fh)
im = d[in_key]
gt = d[out_key]
x, ytrue, ypred = predict(model, x=im, ytrue=gt)
d_res = metrics_computer.compute(ypred=ypred, ytrue=ytrue)
print(f"Scores")
for k in d_res.keys():
    print(f"{k}: {d_res[k]:.2f}")

ims = (x, ypred, ytrue)
titles = ('input', 'prediction', 'target')
fig, axs = plt.subplots(nrows=1,ncols=3, figsize=(8*3,8*1), dpi=200)
for i in range(3):
    axs[i].imshow(ims[i], cmap='gray')
    axs[i].set_title(titles[i])
    axs[i].set_axis_off()
plt.show()

2. On a random test set sample (requires public dataset to be downloaded)

In [ ]:
## Example with swfd_lv128,sc
task_str = 'swfd_lv128,sc'
model = mpm_obj.load_model(task_str=task_str)

## load task and dataset relevant hyperparameters
datasets_parent_dir = '/home/firat/docs/dlbirhoui/parsed_data2'
hpobj = utils.load_model_hyperparameters(task_str=task_str, datasets_parent_dir=datasets_parent_dir)

fname = hpobj.fname_h5
inds_test = hpobj.inds_test
inds_train = hpobj.inds_train
inds_val = hpobj.inds_val
in_key = hpobj.in_key
out_key = hpobj.out_key


metrics_computer = utils.Metrics_Translation()

## predict and evaluate a random sample
i = np.random.choice(inds_test)
with h5py.File(fname, 'r') as fh:
    im = fh[in_key][i,...]
    gt = fh[out_key][i,...]
x, ytrue, ypred = predict(model, x=im, ytrue=gt)
d_res = metrics_computer.compute(ypred=ypred, ytrue=ytrue)
print(f"Scores")
for k in d_res.keys():
    print(f"{k}: {d_res[k]:.2f}")

ims = (x, ypred, ytrue)
titles = ('input', 'prediction', 'target')
fig, axs = plt.subplots(nrows=1,ncols=3, figsize=(8*3,8*1), dpi=200)
for i in range(3):
    axs[i].imshow(ims[i], cmap='gray')
    axs[i].set_title(titles[i])
    axs[i].set_axis_off()
plt.show()